# True-guided Bloch surface wave

A Bloch surface wave (BSW) is localized at the termination of a one-dimensional photonic crystal. On the cover side it decays by total internal reflection; inside the distributed Bragg reflector (DBR) it decays because its propagation constant lies in a photonic stop band.

Experimental BSW platforms often use a finite DBR on a high-index prism, which opens a weak leakage channel for optical coupling. This example instead uses low-index semi-infinite media on both sides. Since the effective index is above both exterior indices, no propagating exterior channel exists and REMSOL returns a true guided mode with real `neff`.

The six-period DBR and termination-layer construction follows the general geometry used in T. Fort et al., [Photonics 9, 561 (2022)](https://doi.org/10.3390/photonics9080561). The indices and thicknesses below are simplified, nondispersive demonstration values rather than a reproduction of that experiment.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import remsol
from remsol import Polarization as pol

WAVELENGTH = 1.55  # um
K0 = 2.0 * np.pi / WAVELENGTH

N_BACKING = 1.0
N_COVER = 1.0
N_HIGH = 2.2
N_LOW = 1.45
N_PERIODS = 6

# Quarter-wave DBR at the design wavelength.
D_HIGH = WAVELENGTH / (4.0 * N_HIGH)
D_LOW = WAVELENGTH / (4.0 * N_LOW)

# A high-index termination layer creates the surface defect state.
D_TERMINATION = 0.30  # um
PLOT_WINDOW = 1.5  # um on each exterior side

## Structure

The layer order is low-index backing, six high/low DBR periods, a high-index termination layer, and low-index cover. The first and last thicknesses set only the plotting windows. All other thicknesses are physical.

In [ ]:
def make_bsw_structure(termination_thickness=D_TERMINATION):
    layers = [remsol.Layer(N_BACKING, PLOT_WINDOW)]
    for _ in range(N_PERIODS):
        layers.extend(
            [
                remsol.Layer(N_HIGH, D_HIGH),
                remsol.Layer(N_LOW, D_LOW),
            ]
        )
    layers.extend(
        [
            remsol.Layer(N_HIGH, float(termination_thickness)),
            remsol.Layer(N_COVER, PLOT_WINDOW),
        ]
    )
    return remsol.MultiLayer(layers)


structure = make_bsw_structure()
index = structure.index()

period = D_HIGH + D_LOW
dbr_end = N_PERIODS * period
surface = dbr_end + D_TERMINATION

fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(index.x, index.n)
ax.axvline(dbr_end, color="tab:orange", linestyle="--", label="termination starts")
ax.axvline(surface, color="black", linestyle="--", label="DBR/cover surface")
ax.set(xlabel="Transverse coordinate x (um)", ylabel="Refractive index")
ax.grid(alpha=0.3)
ax.legend()
plt.show()

## Identify the surface state

For one period, the TE Bloch discriminant is one half of the period transfer-matrix trace. Real Bloch propagation is allowed when `|D| <= 1`; `|D| > 1` is a stop band, where the Bloch wavevector has a nonzero imaginary part.

The finite stack supports several ordinary slab-like modes. The BSW is selected as the real mode whose effective index lies inside the infinite-DBR stop band.

In [ ]:
def te_bloch_discriminant(neff):
    q_high = K0 * np.sqrt(N_HIGH**2 - neff**2 + 0j)
    q_low = K0 * np.sqrt(N_LOW**2 - neff**2 + 0j)
    return (
        np.cos(q_high * D_HIGH) * np.cos(q_low * D_LOW)
        - 0.5
        * (q_high / q_low + q_low / q_high)
        * np.sin(q_high * D_HIGH)
        * np.sin(q_low * D_LOW)
    )


all_te_modes = structure.all_neff(K0, pol.TE)
stop_band_modes = [
    neff for neff in all_te_modes if abs(te_bloch_discriminant(neff)) > 1.0
]

print("All real TE modes:")
for neff in all_te_modes:
    discriminant = te_bloch_discriminant(neff)
    region = "stop band" if abs(discriminant) > 1.0 else "allowed band"
    print(f"  neff={neff:.9f}, |D|={abs(discriminant):.6f} ({region})")

assert len(stop_band_modes) == 1
bsw_neff = stop_band_modes[0]
print(f"\nSelected BSW: neff={bsw_neff:.9f}")

In [ ]:
neff_grid = np.linspace(N_COVER + 1e-3, N_HIGH - 1e-3, 700)
discriminant_grid = np.abs(
    np.asarray([te_bloch_discriminant(neff) for neff in neff_grid])
)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(neff_grid, discriminant_grid, label="|Bloch discriminant|")
ax.axhline(1.0, color="black", linestyle="--", label="stop-band boundary")
for neff in all_te_modes:
    ax.axvline(neff, color="0.75", linewidth=0.8)
ax.axvline(bsw_neff, color="tab:red", linewidth=2.0, label="BSW")
ax.set(
    xlabel="Effective index",
    ylabel="|D|",
    ylim=(0.0, min(2.2, 1.1 * discriminant_grid.max())),
)
ax.grid(alpha=0.3)
ax.legend()
plt.show()

## Termination-layer tuning

The surface state is controlled by the phase accumulated in the termination layer. For this design it enters the stop band near 280 nm and moves deeper into the gap as the termination becomes thicker.

In [ ]:
termination_values = np.linspace(0.28, 0.36, 9)
termination_modes = []
termination_discriminants = []

for thickness in termination_values:
    candidate_modes = make_bsw_structure(thickness).all_neff(K0, pol.TE)
    candidates = [
        neff
        for neff in candidate_modes
        if abs(te_bloch_discriminant(neff)) > 1.0
    ]
    if len(candidates) != 1:
        raise RuntimeError(
            f"Expected one stop-band mode for termination={thickness:.3f} um"
        )
    termination_modes.append(candidates[0])
    termination_discriminants.append(
        abs(te_bloch_discriminant(candidates[0]))
    )

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(1e3 * termination_values, termination_modes, "o-")
axes[0].set(ylabel="BSW effective index")
axes[1].plot(1e3 * termination_values, termination_discriminants, "o-")
axes[1].axhline(1.0, color="black", linestyle="--")
axes[1].set(ylabel="|Bloch discriminant|")
for ax in axes:
    ax.set_xlabel("Termination thickness (nm)")
    ax.grid(alpha=0.3)
plt.show()

## Localized field

Because the selected mode is real, the ordinary `field` method reconstructs it. For TE polarization, `Ey` is the dominant electric component. Its oscillatory envelope grows through the finite DBR toward the surface, peaks in the termination layer, and then decays exponentially into the cover.

In [ ]:
bsw_mode_index = all_te_modes.index(bsw_neff)
field = structure.field(K0, pol.TE, bsw_mode_index)

x = np.asarray(field.x)
ey = np.asarray(field.Ey)
intensity = np.abs(ey) ** 2
n_profile = np.asarray(structure.index().n)

fig, ax = plt.subplots(figsize=(11, 4.5))
ax.plot(x, intensity, color="tab:blue", label="|Ey|^2")
ax.axvspan(dbr_end, surface, color="tab:orange", alpha=0.18, label="termination")
ax.axvline(surface, color="black", linestyle="--", label="surface")
ax.set(xlabel="Transverse coordinate x (um)", ylabel="Normalized |Ey|^2")
ax.grid(alpha=0.3)

index_axis = ax.twinx()
index_axis.plot(x, n_profile, color="0.5", alpha=0.45, linewidth=1.0)
index_axis.set_ylabel("Refractive index", color="0.4")

ax.legend(loc="upper left")
plt.show()

In [ ]:
# Quantitative localization checks.
period_maxima = []
for period_index in range(N_PERIODS):
    start = period_index * period
    stop = (period_index + 1) * period
    in_period = (x >= start) & (x < stop)
    period_maxima.append(np.max(np.abs(ey[in_period])))

peak_position = x[np.argmax(np.abs(ey))]
max_e = np.max(
    np.sqrt(
        np.abs(np.asarray(field.Ex)) ** 2
        + np.abs(np.asarray(field.Ey)) ** 2
        + np.abs(np.asarray(field.Ez)) ** 2
    )
)

print(f"BSW neff: {bsw_neff:.9f}")
print(f"|D| at the BSW: {abs(te_bloch_discriminant(bsw_neff)):.6f}")
print(f"Peak position relative to surface: {peak_position - surface:.3f} um")
print("Field maxima in successive DBR periods:")
print(np.asarray(period_maxima))
print(f"Left plotting-edge amplitude: {abs(ey[0]):.3e}")
print(f"Right plotting-edge amplitude: {abs(ey[-1]):.3e}")

assert bsw_neff > max(N_BACKING, N_COVER)
assert abs(te_bloch_discriminant(bsw_neff)) > 1.0
assert np.all(np.diff(period_maxima) > 0.0)
assert dbr_end < peak_position < surface
assert abs(ey[0]) < 1e-3
assert abs(ey[-1]) < 1e-3
assert np.isclose(max_e, 1.0)

The two confinement mechanisms are distinct: total internal reflection excludes radiation into the homogeneous backing and cover, while the DBR stop band excludes a propagating Bloch wave inside the periodic stack. Consequently this construction has a real effective index and is not fundamentally leaky.

A high-index prism backing would instead provide a propagating coupling channel. With a finite DBR that experimentally useful configuration becomes a weakly leaky resonance and should be treated with an outgoing boundary and the complex solver.